<a href="https://colab.research.google.com/github/francji1/01RAD/blob/main/HW/01RAD_Ex02_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01RAD - Homework 02: Regression to the mean (Galton's heights)

In 1886 Francis Galton compared the heights of 1,078 fathers and their adult sons and noticed that sons of tall fathers tend to be shorter than their fathers, and sons of short fathers taller. He called the phenomenon *regression towards mediocrity*; the name stuck to the whole method (Lecture 1, history). The data are the `fsdata.csv` file from Exercise 01 (heights in inches).

Model of Lecture 1: $Y_i = \beta_0 + \beta_1 x_i + e_i$ with $x_i$ the father's height and $Y_i$ the son's height, independent errors $e_i \sim (0, \sigma^2)$. Tools of Lecture 2 (slides pp. 14–32 of the lecture PDF): standard errors $\widehat{\sigma}(\widehat{\beta}_j) = s_n \sqrt{\delta_j}$, **confidence intervals** (Czech: intervaly spolehlivosti) $\widehat{\beta}_j \pm t_{1-\alpha/2}(n-2)\,\widehat{\sigma}(\widehat{\beta}_j)$ and the **tests** (testy hypotéz) with statistics $T_j = (\widehat{\beta}_j - b)/\widehat{\sigma}(\widehat{\beta}_j)$ compared with $t_{1-\alpha/2}(n-2)$.

## Submission
Work through the tasks in this notebook (code plus short written answers in markdown cells; Python or R, as you prefer). Submit through GitHub **before the next exercise**: fork `francji1/01RAD`, save the notebook into the `HW/` folder as `01RAD_Ex02_HW_<Surname>.ipynb`, with your surname in place of `<Surname>` and without the brackets, for example `01RAD_Ex02_HW_Novak.ipynb` (team: surnames joined by an underscore), and open a pull request to `main`. The pull request should add your one file only; do not modify or delete other files. A selected student solution, and a reference solution if needed, will be published in `HW/` a week later.

Useful calls: `stats.t.ppf(1 - alpha/2, df)` for the $t$ quantile, `stats.t.sf(abs(T), df)` and `stats.t.cdf(T, df)` for tail probabilities, `model.conf_int(alpha)`, `model.t_test('father = 1')`, `model.predict(pd.DataFrame({'father': [...]}))`, `np.corrcoef`, `df.sample(n, random_state=...)`.

## Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', palette='deep')

url_fsdata = 'https://raw.githubusercontent.com/francji1/01RAD/main/data/fsdata.csv'
fsdata = pd.read_csv(url_fsdata, index_col=0)
print(f'Rows: {fsdata.shape[0]}, columns: {fsdata.shape[1]}')
fsdata.describe().T

Rows: 1078, columns: 2


,count,mean,std,min,25%,50%,75%,max
father,1078.0,67.687097,2.744868,59.00800,65.787735,67.76660,69.60298,75.43393
son,1078.0,68.684070,2.814702,58.50708,66.931232,68.61582,70.46597,78.36479


## Task 01 - Fit and recap
Fit the regression of `son` on `father`. Compute $\widehat{\beta}_0$, $\widehat{\beta}_1$, $s_n$ and both standard errors by hand (formulas of Lectures 1 and 2) and check them against `statsmodels` (R: `summary(lm(son ~ father))`). Interpret $\widehat{\beta}_1$ in one sentence.

In [2]:
import numpy as np
import statsmodels.api as sm

x = fsdata['father']
y = fsdata['son']
n = len(x)

mean_x = np.mean(x)
mean_y = np.mean(y)

S_xx = np.sum((x - mean_x)**2)
S_xy = np.sum((x - mean_x) * (y - mean_y))

beta_1_hat = S_xy / S_xx
beta_0_hat = mean_y - beta_1_hat * mean_x

y_hat = beta_0_hat + beta_1_hat * x
residuals = y - y_hat

RSS = np.sum(residuals**2)
s_n_squared = RSS / (n - 2)
s_n = np.sqrt(s_n_squared)

se_beta_1 = s_n / np.sqrt(S_xx)
se_beta_0 = s_n * np.sqrt((1/n) + (mean_x**2 / S_xx))

print("Ruční výpočet:")
print(f"beta_0_hat = {beta_0_hat:.4f}, SE(beta_0_hat) = {se_beta_0:.4f}")
print(f"beta_1_hat = {beta_1_hat:.4f}, SE(beta_1_hat) = {se_beta_1:.4f}")
print(f"s_n = {s_n:.4f}\n")

X_sm = sm.add_constant(x)
model = sm.OLS(y, X_sm).fit()
print("Výpočet pomocí statsmodels:")
print(model.summary().tables[1])

print("\n--- Odpověď ---")
print("Interpretace beta_1: Otec vyšší o 1 palec znamená syna v průměru vyššího o ~0.51 palce.")

Ruční výpočet:
beta_0_hat = 33.8866, SE(beta_0_hat) = 1.8324
beta_1_hat = 0.5141, SE(beta_1_hat) = 0.0270
s_n = 2.4366

Výpočet pomocí statsmodels:
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         33.8866      1.832     18.493      0.000      30.291      37.482
father         0.5141      0.027     19.006      0.000       0.461       0.567

--- Odpověď ---
Interpretace beta_1: Otec vyšší o 1 palec znamená syna v průměru vyššího o ~0.51 palce.


## Task 02 - Confidence intervals
Compute the 95 % and the 99 % confidence intervals for $\beta_0$ and $\beta_1$ by hand, using the $t$ quantile, and compare with `conf_int` (R: `confint`). Why is the 99 % interval wider? Compare the widths of the intervals for $\beta_0$ and $\beta_1$ in absolute terms and relative to the estimates: which parameter is estimated more precisely? Relate your answer to the remark in Lecture 2.

In [3]:
from scipy import stats

df = n - 2

t_95 = stats.t.ppf(0.975, df)
t_99 = stats.t.ppf(0.995, df)

ci95_beta_0 = (beta_0_hat - t_95 * se_beta_0, beta_0_hat + t_95 * se_beta_0)
ci95_beta_1 = (beta_1_hat - t_95 * se_beta_1, beta_1_hat + t_95 * se_beta_1)

ci99_beta_0 = (beta_0_hat - t_99 * se_beta_0, beta_0_hat + t_99 * se_beta_0)
ci99_beta_1 = (beta_1_hat - t_99 * se_beta_1, beta_1_hat + t_99 * se_beta_1)

print("Ručně vypočítané intervaly spolehlivosti:")
print(f"95% CI pro beta_0: {ci95_beta_0}")
print(f"95% CI pro beta_1: {ci95_beta_1}\n")

print("Intervaly ze statsmodels (95%):")
print(model.conf_int(alpha=0.05))

width_95_beta_0 = ci95_beta_0[1] - ci95_beta_0[0]
width_95_beta_1 = ci95_beta_1[1] - ci95_beta_1[0]

print("\n--- Odpověď ---")
print("1) 99% CI je širší kvůli požadavku na vyšší pravděpodobnost pokrytí.")
print("2) Relativně (šířka/odhad) je přesnější beta_0, absolutně je užší beta_1.")

Ručně vypočítané intervaly spolehlivosti:
95% CI pro beta_0: (np.float64(30.291212566757814), np.float64(37.481996141398035))
95% CI pro beta_1: (np.float64(0.46101878906416827), np.float64(0.5671672881824467))

Intervaly ze statsmodels (95%):
                0          1
const   30.291213  37.481996
father   0.461019   0.567167

--- Odpověď ---
1) 99% CI je širší kvůli požadavku na vyšší pravděpodobnost pokrytí.
2) Relativně (šířka/odhad) je přesnější beta_0, absolutně je užší beta_1.


## Task 03 - Is the father's height informative?
Test $H_0: \beta_1 = 0$ against $H_1: \beta_1 \neq 0$ at $\alpha = 0.05$: report $T_1$, the critical value $t_{0.975}(n-2)$, the $p$-value and the decision. What does the rejection say about the relationship, and what does it *not* say (think of the lecture's warning about linearity)?

In [4]:
b_1_H0 = 0
T_1 = (beta_1_hat - b_1_H0) / se_beta_1
critical_value = t_95
p_value = 2 * stats.t.sf(np.abs(T_1), df)

print(f"Testová statistika T_1: {T_1:.4f}")
print(f"Kritická hodnota t_0.975({df}): {critical_value:.4f}")
print(f"p-hodnota: {p_value:.4e}")

print("\n--- Odpověď ---")
print("Závěr: Zamítáme H0, výška otce je informativní.")
print("Co to neříká: Neříká to, že vztah je čistě lineární nebo deterministický.")

Testová statistika T_1: 19.0062
Kritická hodnota t_0.975(1076): 1.9622
p-hodnota: 1.1213e-69

--- Odpověď ---
Závěr: Zamítáme H0, výška otce je informativní.
Co to neříká: Neříká to, že vztah je čistě lineární nebo deterministický.


## Task 04 - Regression to the mean
If sons were on average exactly as tall as their fathers plus a constant, the slope would be $\beta_1 = 1$. Test $H_0: \beta_1 = 1$ against $H_1: \beta_1 \neq 1$ (the test with a given candidate value $b_1$ from Lecture 2) and also against the **one-sided alternative** (jednostranná alternativa) $H_1: \beta_1 < 1$: the lecture treats two-sided alternatives, for the one-sided one use the same statistic $T = (\widehat{\beta}_1 - 1)/\widehat{\sigma}(\widehat{\beta}_1)$, reject when $T < -t_{1-\alpha}(n-2)$, and the $p$-value is $P(t(n-2) < T)$. Check the duality of the two-sided test with the 95 % confidence interval from Task 02. Explain in your own words why Galton spoke of *regression* (regrese k průměru) and what the estimated slope says about sons of very tall and very short fathers.

In [5]:
b_1_H0_task4 = 1
T_task4 = (beta_1_hat - b_1_H0_task4) / se_beta_1
p_value_two_sided = 2 * stats.t.sf(np.abs(T_task4), df)

t_90 = stats.t.ppf(0.95, df)
critical_value_one_sided = -t_90
p_value_one_sided = stats.t.cdf(T_task4, df)

print("Jednostranný test (beta_1 < 1):")
print(f"Testová statistika T: {T_task4:.4f}")
print(f"Kritická hodnota: {critical_value_one_sided:.4f}")
print(f"p-hodnota: {p_value_one_sided:.4e}")

print("\n--- Odpověď ---")
print("Dualita s CI: 1 neleží v 95% CI pro beta_1, proto zamítáme H0 (beta_1 = 1) na hladině 0.05.")
print("Regrese k průměru: Potomci extrémně vysokých/malých otců mají výšku blíže k celkovému průměru.")

Jednostranný test (beta_1 < 1):
Testová statistika T: -17.9641
Kritická hodnota: -1.6463
p-hodnota: 1.3048e-63

--- Odpověď ---
Dualita s CI: 1 neleží v 95% CI pro beta_1, proto zamítáme H0 (beta_1 = 1) na hladině 0.05.
Regrese k průměru: Potomci extrémně vysokých/malých otců mají výšku blíže k celkovému průměru.


## Task 05 - The intercept
Test $H_0: \beta_0 = 0$. Then write $\widehat{\beta}_0 = \overline{y} - \widehat{\beta}_1 \overline{x}$, plug in the sample means and explain why the intercept is far from zero even though sons are on average only about one inch taller than their fathers. Would a model through the origin make sense here?

In [6]:
b_0_H0 = 0
T_0 = (beta_0_hat - b_0_H0) / se_beta_0
p_value_0 = 2 * stats.t.sf(np.abs(T_0), df)

print(f"Testová statistika T_0: {T_0:.4f}, p-hodnota: {p_value_0:.4e}")

print("\n--- Odpověď ---")
print(f"beta_0_hat = {mean_y:.2f} - {beta_1_hat:.4f} * {mean_x:.2f} = {beta_0_hat:.4f}")
print("Intercept kompenzuje sklon beta_1 < 1 pro zachování celkových průměrů.")
print("Model bez interceptu nemá smysl (extrapolace k výšce otce 0 nedává smysl).")

Testová statistika T_0: 18.4935, p-hodnota: 1.6040e-66

--- Odpověď ---
beta_0_hat = 68.68 - 0.5141 * 67.69 = 33.8866
Intercept kompenzuje sklon beta_1 < 1 pro zachování celkových průměrů.
Model bez interceptu nemá smysl (extrapolace k výšce otce 0 nedává smysl).


## Task 06 - Point predictions
Predict the mean height of sons whose fathers are 60, 64, 68, 72 and 76 inches tall and tabulate the difference *predicted son minus father*. For which father's height do the two coincide? (Confidence and prediction intervals for these predictions come in Lecture 4; here point predictions are enough.)

In [7]:
import pandas as pd
fathers_heights = np.array([60, 64, 68, 72, 76])
predicted_sons = beta_0_hat + beta_1_hat * fathers_heights
differences = predicted_sons - fathers_heights

predictions_df = pd.DataFrame({'Father': fathers_heights, 'Predicted_Son': predicted_sons, 'Difference': differences})
print(predictions_df)

intersection_height = beta_0_hat / (1 - beta_1_hat)
print("\n--- Odpověď ---")
print(f"Predikce a výška otce se shodují při {intersection_height:.2f} palcích.")

   Father  Predicted_Son  Difference
0      60      64.732187    4.732187
1      64      66.788559    2.788559
2      68      68.844931    0.844931
3      72      70.901303   -1.098697
4      76      72.957675   -3.042325

--- Odpověď ---
Predikce a výška otce se shodují při 69.74 palcích.


## Task 07 - Reverse regression
Fit the reverse regression of `father` on `son` and report its slope. Verify numerically that the product of the two slopes equals the square of the sample correlation coefficient $r$ of the two heights (`np.corrcoef`), and explain why both slopes are smaller than 1. Is regression to the mean a causal effect of fathers on sons, or a statistical property of the two distributions?

In [8]:
X_rev = sm.add_constant(y)
model_rev = sm.OLS(x, X_rev).fit()
beta_1_rev_hat = model_rev.params.iloc[1]

r = np.corrcoef(x, y)[0, 1]

print(f"Sklon původní (son na father): {beta_1_hat:.4f}")
print(f"Sklon reverzní (father na son): {beta_1_rev_hat:.4f}")
print(f"Součin sklonů: {beta_1_hat * beta_1_rev_hat:.4f}, r^2: {r**2:.4f}")

print("\n--- Odpověď ---")
print("Oba sklony < 1 ukazují regresi k průměru oběma směry. Rovnost beta_1 * beta_1_rev = r^2 potvrzuje přímou vazbu sklonu na korelaci.")
print("Regrese k průměru je statistická vlastnost dvou nedokonale korelovaných proměnných (r < 1), nikoliv kauzální efekt.")
print("Neznamená to, že otec fyziologicky 'táhne' syna k průměru. Znamená to jen, že pokud vybereme extrémní hodnotu (např. extrémně vysokého syna), pravděpodobně k tomu přispěla i souhra náhodných faktorů. Hodnota druhé veličiny (výška otce) pak bude přirozeně méně extrémní, tedy blíže svému průměru.")

Sklon původní (son na father): 0.5141
Sklon reverzní (father na son): 0.4889
Součin sklonů: 0.2513, r^2: 0.2513

--- Odpověď ---
Oba sklony < 1 ukazují regresi k průměru oběma směry.
Jde o statistickou vlastnost vlivu náhody u okrajových hodnot, nikoliv o kauzální sílu.


## Task 08 - Sample size and precision
Draw random subsamples of 30, 100 and 300 father-son pairs (fix the random seed), fit the model in each and compare the widths of the 95 % intervals for $\beta_1$ with the full-sample interval. Explain the pattern through $\mathrm{Var}(\widehat{\beta}_1) = \sigma^2 / S_{xx}$. *Optional:* repeat the subsampling with $n = 100$ five hundred times, treat the full-sample $\widehat{\beta}_1$ as the true value and estimate how often the 95 % interval covers it.

In [9]:
sample_sizes = [30, 100, 300]
np.random.seed(42)

print(f"Šířka intervalu celý dataset: {ci95_beta_1[1] - ci95_beta_1[0]:.4f}\n")

for size in sample_sizes:
    sample = fsdata.sample(n=size, random_state=42)
    xs, ys = sample['father'], sample['son']
    mod = sm.OLS(ys, sm.add_constant(xs)).fit()
    ci = mod.conf_int(alpha=0.05).iloc[1]
    S_xx_s = np.sum((xs - np.mean(xs))**2)
    print(f"n={size}, šířka CI: {ci[1] - ci[0]:.4f}, S_xx: {S_xx_s:.2f}")

print("\n--- Odpověď ---")
print("S rostoucím 'n' roste S_xx. Podle Var(beta_1) = sigma^2 / S_xx to zmenšuje rozptyl a zužuje CI.")

Šířka intervalu celý dataset: 0.1061

n=30, šířka CI: 0.6145, S_xx: 198.06
n=100, šířka CI: 0.3388, S_xx: 697.04
n=300, šířka CI: 0.2067, S_xx: 2034.83

--- Odpověď ---
S rostoucím 'n' roste S_xx. Podle Var(beta_1) = sigma^2 / S_xx to zmenšuje rozptyl a zužuje CI.
